[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_04/notebook_4_3_feature_engineering.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 4.3: Feature Engineering for Healthcare ML

**Chapter 4: Data Preparation**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Create clinically meaningful derived features
2. Engineer temporal features from time series data
3. Design interaction features that capture clinical relationships
4. Implement domain-specific transformations
5. Evaluate feature importance and select relevant features

## Clinical Context

**Raw data is rarely optimal for ML**:
- Labs have clinical meaning only in context (age, sex, comorbidities)
- Trends matter more than single values (glucose rising vs stable)
- Clinical scores aggregate multiple variables (Framingham, MELD, APACHE)
- Ratios capture physiological relationships (BUN/Cr, ANC)

**Good features:**
- Encode clinical knowledge
- Are interpretable to clinicians
- Improve model performance
- Generalize across institutions

---

## Setup

In [1]:
# ── Setup (JupyterLite) ─────────────────────────────────────────────────────
import sys
if sys.platform == "emscripten":          # JupyterLite / Pyodide
    import micropip
    await micropip.install(["seaborn"])

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Libraries imported")

✓ Libraries imported


## 1. Generate Base Clinical Dataset

In [3]:
def generate_clinical_data(n_patients=1000):
    """
    Generate synthetic patient data for acute kidney injury (AKI) prediction.
    """

    # Demographics
    age = np.random.normal(65, 15, n_patients)
    age = np.clip(age, 18, 100)

    sex = np.random.choice([0, 1], n_patients)  # 0=F, 1=M

    # Baseline kidney function
    baseline_cr = np.random.gamma(2, 0.5, n_patients)  # Baseline creatinine
    baseline_cr = np.clip(baseline_cr, 0.5, 3.0)

    # Current labs
    creatinine = baseline_cr + np.random.normal(0, 0.3, n_patients)  # Current creatinine
    creatinine = np.clip(creatinine, 0.5, 8.0)

    bun = creatinine * 10 + np.random.normal(0, 5, n_patients)  # BUN correlates with Cr
    bun = np.clip(bun, 5, 100)

    potassium = 4.0 + np.random.normal(0, 0.5, n_patients)
    potassium = np.clip(potassium, 2.5, 7.0)

    # Vitals
    sbp = np.random.normal(130, 20, n_patients)
    sbp = np.clip(sbp, 80, 200)

    heart_rate = np.random.normal(80, 15, n_patients)
    heart_rate = np.clip(heart_rate, 50, 150)

    # Urine output (mL/kg/hr)
    weight = 70 + np.random.normal(0, 15, n_patients)
    weight = np.clip(weight, 40, 150)

    urine_24h = np.random.normal(2000, 500, n_patients)  # mL per 24 hours
    urine_24h = np.clip(urine_24h, 200, 4000)

    # Comorbidities
    diabetes = np.random.choice([0, 1], n_patients, p=[0.7, 0.3])
    hypertension = np.random.choice([0, 1], n_patients, p=[0.6, 0.4])
    heart_failure = np.random.choice([0, 1], n_patients, p=[0.85, 0.15])

    # Outcome: AKI risk rises with Cr increase >0.2 mg/dL or >1.3x baseline (lower cutoffs than
    # KDIGO, which defines AKI as a rise >=0.3 mg/dL within 48h or >=1.5x baseline), then AKI is sampled
    cr_increase = creatinine - baseline_cr
    cr_ratio = creatinine / baseline_cr

    aki_risk = (
        0.5 * (cr_increase > 0.2) +
        0.3 * (cr_ratio > 1.3) +
        0.2 * (urine_24h < 1000) +
        0.1 * diabetes +
        0.1 * heart_failure +
        0.05 * (age - 65) / 15
    )

    prob = 1 / (1 + np.exp(-aki_risk + 0.5))
    aki = (np.random.rand(n_patients) < prob).astype(int)

    df = pd.DataFrame({
        'patient_id': [f'P{i:04d}' for i in range(n_patients)],
        'age': age,
        'sex': sex,
        'weight': weight,
        'baseline_cr': baseline_cr,
        'creatinine': creatinine,
        'bun': bun,
        'potassium': potassium,
        'sbp': sbp,
        'heart_rate': heart_rate,
        'urine_24h': urine_24h,
        'diabetes': diabetes,
        'hypertension': hypertension,
        'heart_failure': heart_failure,
        'aki': aki
    })

    return df

df = generate_clinical_data(1000)
print(f"Generated {len(df)} patient records")
print(f"AKI prevalence: {df['aki'].mean():.1%}")
print(f"\nBase features: {len(df.columns) - 2} (excluding patient_id and outcome)")
df.head()

Generated 1000 patient records
AKI prevalence: 40.4%

Base features: 13 (excluding patient_id and outcome)


,patient_id,age,sex,weight,baseline_cr,creatinine,bun,potassium,sbp,heart_rate,urine_24h,diabetes,hypertension,heart_failure,aki
0,P0000,72.450712,0,83.366058,0.619063,0.993949,8.262489,4.030578,80.000000,87.419370,2123.140906,0,0,0,0
1,P0001,62.926035,1,67.689400,0.958319,0.895612,21.513822,4.380011,139.335104,103.624230,1779.718233,0,0,0,1
2,P0002,74.715328,0,58.190111,0.500000,0.660046,6.397702,3.802922,145.800308,91.382014,981.965901,1,0,0,1
3,P0003,87.845448,1,50.183265,0.500000,0.500000,6.221055,3.612849,95.999605,82.122982,1510.373833,0,0,0,0
4,P0004,61.487699,0,73.759475,0.500000,0.500000,5.000000,4.405432,128.660553,105.102903,1994.410442,0,1,0,0


## 2. Clinical Ratios and Derived Features

In [4]:
def engineer_clinical_ratios(df):
    """
    Create clinically meaningful ratio features.
    """
    df_fe = df.copy()

    # 1. BUN/Creatinine ratio (pre-renal azotemia marker)
    # Normal: 10-20, >20 suggests pre-renal, <10 suggests intrinsic renal
    df_fe['bun_cr_ratio'] = df_fe['bun'] / df_fe['creatinine']

    # 2. Creatinine change from baseline
    df_fe['cr_delta'] = df_fe['creatinine'] - df_fe['baseline_cr']
    df_fe['cr_pct_change'] = (df_fe['creatinine'] - df_fe['baseline_cr']) / df_fe['baseline_cr'] * 100

    # 3. Estimated GFR (MDRD 4-variable equation, without the race term; CKD-EPI is the current standard)
    # GFR = 175 * (Cr)^-1.154 * (Age)^-0.203 * (0.742 if female)
    gfr = 175 * (df_fe['creatinine'] ** -1.154) * (df_fe['age'] ** -0.203)
    gfr = np.where(df_fe['sex'] == 0, gfr * 0.742, gfr)  # Adjust for female
    df_fe['egfr'] = gfr

    # 4. Urine output per kg per hour
    df_fe['urine_ml_kg_hr'] = df_fe['urine_24h'] / (df_fe['weight'] * 24)

    # 5. Pulse pressure (marker of vascular stiffness)
    # Assuming DBP ~ 0.6 * SBP for simplicity
    df_fe['pulse_pressure'] = df_fe['sbp'] * 0.4

    # 6. Mean arterial pressure
    # MAP = SBP/3 + 2*DBP/3 (approximation)
    df_fe['map'] = df_fe['sbp'] / 3 + df_fe['sbp'] * 0.6 * 2 / 3

    # 7. Shock index = heart rate / systolic BP (normal ~0.5-0.7; >0.9 suggests hemodynamic instability)
    df_fe['shock_index'] = df_fe['heart_rate'] / df_fe['sbp']

    return df_fe

df_ratios = engineer_clinical_ratios(df)

print("Clinical Ratios Created:")
new_features = ['bun_cr_ratio', 'cr_delta', 'cr_pct_change', 'egfr', 'urine_ml_kg_hr', 'pulse_pressure', 'map', 'shock_index']
print(df_ratios[new_features].describe().T)

# Show clinical interpretation
print("\n" + "="*70)
print("CLINICAL INTERPRETATION")
print("="*70)
print("BUN/Cr ratio:")
print(f"  >20 (pre-renal): {(df_ratios['bun_cr_ratio'] > 20).sum()} patients ({(df_ratios['bun_cr_ratio'] > 20).mean():.1%})")
print(f"  <10 (intrinsic): {(df_ratios['bun_cr_ratio'] < 10).sum()} patients ({(df_ratios['bun_cr_ratio'] < 10).mean():.1%})")

print("\neGFR stages (CKD):")
print(f"  Stage 1 (>90): {(df_ratios['egfr'] > 90).sum()} patients")
print(f"  Stage 2 (60-90): {((df_ratios['egfr'] >= 60) & (df_ratios['egfr'] <= 90)).sum()} patients")
print(f"  Stage 3 (30-60): {((df_ratios['egfr'] >= 30) & (df_ratios['egfr'] < 60)).sum()} patients")
print(f"  Stage 4 (15-30): {((df_ratios['egfr'] >= 15) & (df_ratios['egfr'] < 30)).sum()} patients")
print(f"  Stage 5 (<15): {(df_ratios['egfr'] < 15).sum()} patients")

print("\nShock index (HR/SBP):")
print(f"  >0.9 (possible instability): {(df_ratios['shock_index'] > 0.9).sum()} patients ({(df_ratios['shock_index'] > 0.9).mean():.1%})")

print("\nUrine output:")
print(f"  Oliguria (<0.5 mL/kg/hr): {(df_ratios['urine_ml_kg_hr'] < 0.5).sum()} patients ({(df_ratios['urine_ml_kg_hr'] < 0.5).mean():.1%})")
print("="*70)

Clinical Ratios Created:
                 count       mean        std        min        25%        50%  \
bun_cr_ratio    1000.0  11.214326   4.596972   3.367351   8.266291  10.000000   
cr_delta        1000.0   0.046897   0.259596  -0.776525  -0.096146   0.000000   
cr_pct_change   1000.0   9.040440  31.203046 -59.798060  -8.695898   0.000000   
egfr            1000.0  81.942925  46.057468  12.869560  43.447126  71.439938   
urine_ml_kg_hr  1000.0   1.233290   0.431704   0.249035   0.927627   1.173347   
pulse_pressure  1000.0  51.714313   8.136274  32.000000  46.352469  51.664531   
map             1000.0  94.809574  14.916503  58.666667  84.979526  94.718307   

                       75%         max  
bun_cr_ratio     13.183295   35.511743  
cr_delta          0.209838    0.829794  
cr_pct_change    20.910476  165.958777  
egfr            120.441687  201.463848  
urine_ml_kg_hr    1.478220    3.227130  
pulse_pressure   57.272341   75.832440  
map             104.999292  139.026139 

## 3. Interaction Features

In [5]:
def engineer_interactions(df):
    """
    Create interaction features that capture clinical synergies.
    """
    df_int = df.copy()

    # 1. Age × Comorbidity burden
    df_int['comorbidity_count'] = df_int['diabetes'] + df_int['hypertension'] + df_int['heart_failure']
    df_int['age_x_comorbidity'] = df_int['age'] * df_int['comorbidity_count']

    # 2. Diabetes × Creatinine (diabetes worsens kidney injury)
    df_int['diabetes_x_cr'] = df_int['diabetes'] * df_int['creatinine']

    # 3. Elderly with low urine output (high risk)
    df_int['elderly_oliguria'] = ((df_int['age'] > 70) & (df_int['urine_ml_kg_hr'] < 0.5)).astype(int)

    # 4. Hypotension + Tachycardia: a binary flag, NOT the shock index (the continuous HR/SBP ratio above)
    df_int['shock_indicator'] = ((df_int['sbp'] < 100) & (df_int['heart_rate'] > 100)).astype(int)

    # 5. High potassium + Low GFR (dangerous combination)
    df_int['hyperkalemia_risk'] = ((df_int['potassium'] > 5.5) & (df_int['egfr'] < 30)).astype(int)

    return df_int

df_interactions = engineer_interactions(df_ratios)

print("Interaction Features Created:")
interaction_features = ['comorbidity_count', 'age_x_comorbidity', 'diabetes_x_cr',
                        'elderly_oliguria', 'shock_indicator', 'hyperkalemia_risk']

for feat in interaction_features:
    if df_interactions[feat].dtype in [int, np.int64]:
        # Binary or count feature
        print(f"\n{feat}:")
        print(f"  Count: {df_interactions[feat].sum()}")
        print(f"  Rate: {df_interactions[feat].mean():.1%}")

        # Association with outcome
        aki_rate_positive = df_interactions[df_interactions[feat] == 1]['aki'].mean() if feat in ['elderly_oliguria', 'shock_indicator', 'hyperkalemia_risk'] else None
        aki_rate_negative = df_interactions[df_interactions[feat] == 0]['aki'].mean() if feat in ['elderly_oliguria', 'shock_indicator', 'hyperkalemia_risk'] else None

        if aki_rate_positive is not None and aki_rate_negative is not None:
            print(f"  AKI rate when present: {aki_rate_positive:.1%}")
            print(f"  AKI rate when absent: {aki_rate_negative:.1%}")
            if aki_rate_negative > 0:
                print(f"  Relative risk: {aki_rate_positive / aki_rate_negative:.2f}x")

Interaction Features Created:

comorbidity_count:
  Count: 823
  Rate: 82.3%

elderly_oliguria:
  Count: 9
  Rate: 0.9%
  AKI rate when present: 33.3%
  AKI rate when absent: 40.5%
  Relative risk: 0.82x

shock_indicator:
  Count: 5
  Rate: 0.5%
  AKI rate when present: 20.0%
  AKI rate when absent: 40.5%
  Relative risk: 0.49x

hyperkalemia_risk:
  Count: 0
  Rate: 0.0%
  AKI rate when present: nan%
  AKI rate when absent: 40.4%
  Relative risk: nanx


## 4. Non-Linear Transformations

In [6]:
def create_nonlinear_features(df):
    """
    Create polynomial and log transformations.
    """
    df_nonlin = df.copy()

    # 1. Quadratic terms (U-shaped relationships)
    # Both very low and very high potassium are dangerous
    df_nonlin['potassium_sq'] = df_nonlin['potassium'] ** 2

    # 2. Log transforms (skewed distributions)
    # Creatinine is often log-normally distributed
    df_nonlin['log_creatinine'] = np.log(df_nonlin['creatinine'] + 1)
    df_nonlin['log_bun'] = np.log(df_nonlin['bun'] + 1)

    # 3. Square root (moderate skewness)
    df_nonlin['sqrt_urine'] = np.sqrt(df_nonlin['urine_24h'])

    # 4. Reciprocal (inverse relationships)
    df_nonlin['reciprocal_egfr'] = 1 / (df_nonlin['egfr'] + 1)  # Avoid division by zero

    return df_nonlin

df_nonlinear = create_nonlinear_features(df_interactions)

# Visualize transformations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original creatinine
ax = axes[0, 0]
ax.hist(df_nonlinear['creatinine'], bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('Creatinine (mg/dL)')
ax.set_ylabel('Count')
ax.set_title('Original Creatinine (Right-Skewed)', fontweight='bold')
ax.grid(True, alpha=0.3)

# Log-transformed creatinine
ax = axes[0, 1]
ax.hist(df_nonlinear['log_creatinine'], bins=50, edgecolor='black', alpha=0.7, color='coral')
ax.set_xlabel('Log(Creatinine + 1)')
ax.set_ylabel('Count')
ax.set_title('Log-Transformed Creatinine (More Normal)', fontweight='bold')
ax.grid(True, alpha=0.3)

# Potassium vs outcome
ax = axes[1, 0]
for aki_status in [0, 1]:
    subset = df_nonlinear[df_nonlinear['aki'] == aki_status]
    ax.scatter(subset['potassium'], subset['potassium_sq'],
              alpha=0.5, s=20, label=f'AKI={aki_status}')
ax.set_xlabel('Potassium (mEq/L)')
ax.set_ylabel('Potassium²')
ax.set_title('Potassium Quadratic Term', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# eGFR vs reciprocal
ax = axes[1, 1]
ax.scatter(df_nonlinear['egfr'], df_nonlinear['reciprocal_egfr'], alpha=0.5, s=20)
ax.set_xlabel('eGFR (mL/min/1.73m²)')
ax.set_ylabel('1 / (eGFR + 1)')
ax.set_title('Reciprocal eGFR (Linear → Non-linear)', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Non-linear transformations created to capture complex relationships.")

Non-linear transformations created to capture complex relationships.


## 5. Feature Selection and Importance

In [7]:
# Prepare datasets for comparison
def prepare_for_modeling(df, feature_cols, outcome_col='aki'):
    """
    Prepare dataset for modeling.
    """
    X = df[feature_cols].copy()
    y = df[outcome_col].copy()

    # Handle any remaining NaNs
    X = X.fillna(X.median())

    return X, y

# Base features (original raw features only)
base_features = ['age', 'sex', 'weight', 'baseline_cr', 'creatinine', 'bun',
                'potassium', 'sbp', 'heart_rate', 'urine_24h',
                'diabetes', 'hypertension', 'heart_failure']

X_base, y = prepare_for_modeling(df, base_features)

# Engineered features
engineered_features = base_features + [
    'bun_cr_ratio', 'cr_delta', 'cr_pct_change', 'egfr', 'urine_ml_kg_hr',
    'shock_index', 'comorbidity_count', 'age_x_comorbidity', 'diabetes_x_cr',
    'elderly_oliguria', 'shock_indicator', 'hyperkalemia_risk',
    'log_creatinine', 'log_bun', 'potassium_sq', 'reciprocal_egfr'
]

X_engineered, y = prepare_for_modeling(df_nonlinear, engineered_features)

# Train models
X_base_train, X_base_test, y_train, y_test = train_test_split(X_base, y, test_size=0.3, random_state=42)
X_eng_train, X_eng_test, _, _ = train_test_split(X_engineered, y, test_size=0.3, random_state=42)

# Base model
clf_base = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5)
clf_base.fit(X_base_train, y_train)
y_pred_base = clf_base.predict_proba(X_base_test)[:, 1]
auroc_base = roc_auc_score(y_test, y_pred_base)

# Engineered features model
clf_eng = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5)
clf_eng.fit(X_eng_train, y_train)
y_pred_eng = clf_eng.predict_proba(X_eng_test)[:, 1]
auroc_eng = roc_auc_score(y_test, y_pred_eng)

print("="*70)
print("FEATURE ENGINEERING IMPACT")
print("="*70)
print(f"Base features only: AUROC = {auroc_base:.3f}")
print(f"With engineered features: AUROC = {auroc_eng:.3f}")
print(f"Improvement: {(auroc_eng - auroc_base):.3f} ({(auroc_eng - auroc_base)/auroc_base * 100:.1f}% relative)")

# Is the difference larger than test-set noise? Paired bootstrap over test patients
rng = np.random.default_rng(0)
diffs = []
for _ in range(1000):
    idx = rng.integers(0, len(y_test), len(y_test))
    yt = y_test.values[idx]
    if yt.min() != yt.max():
        diffs.append(roc_auc_score(yt, y_pred_eng[idx]) - roc_auc_score(yt, y_pred_base[idx]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"95% bootstrap CI for the difference: [{lo:+.3f}, {hi:+.3f}]")
print("="*70)

if max(auroc_base, auroc_eng) < 0.6:
    print("\n⚠️  Both models are close to chance (AUROC 0.5), so this comparison says little.")
    print("   Why: the synthetic generator links AKI only weakly to the features (sampled AKI")
    print("   probabilities span only about 0.35-0.70), so even a perfect model would score modestly.")
    print("   Exercise: increase the weights in aki_risk (e.g. x5) and re-run the comparison.")
if lo > 0:
    print("\n✓ Engineered features improved AUROC beyond test-set noise.")
elif hi < 0:
    print("\n✗ Engineered features made AUROC WORSE beyond test-set noise.")
else:
    print("\n➖ No clear difference: the CI includes zero.")

FEATURE ENGINEERING IMPACT
Base features only: AUROC = 0.492
With engineered features: AUROC = 0.535
Improvement: 0.043 (8.6% relative)


95% bootstrap CI for the difference: [-0.030, +0.116]

⚠️  Both models are close to chance (AUROC 0.5), so this comparison says little.
   Why: the synthetic generator links AKI only weakly to the features (sampled AKI
   probabilities span only about 0.35-0.70), so even a perfect model would score modestly.
   Exercise: increase the weights in aki_risk (e.g. x5) and re-run the comparison.

➖ No clear difference: the CI includes zero.


In [8]:
# Feature importance analysis
feature_importance = pd.DataFrame({
    'feature': engineered_features,
    'importance': clf_eng.feature_importances_
}).sort_values('importance', ascending=False)

# Visualize top 20 features
fig, ax = plt.subplots(figsize=(10, 8))
top_features = feature_importance.head(20)

# Color code: green for engineered, blue for raw
colors = ['green' if feat not in base_features else 'steelblue' for feat in top_features['feature']]

ax.barh(top_features['feature'], top_features['importance'], color=colors)
ax.set_xlabel('Feature Importance')
ax.set_title('Top 20 Features for AKI Prediction\n(Green = Engineered, Blue = Raw)', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

# Count engineered vs raw in top features
top_10_engineered = sum([feat not in base_features for feat in feature_importance.head(10)['feature']])
print(f"\n💡 {top_10_engineered}/10 of top features are engineered.")
if auroc_eng < 0.6:
    print("   Caution: with AUROC near chance, these importances mostly reflect noise the trees fit.")


Top 10 Most Important Features:
          feature  importance
              sbp    0.122067
    cr_pct_change    0.098716
        urine_24h    0.078947
         cr_delta    0.075078
              age    0.066690
           weight    0.059917
       heart_rate    0.054074
   urine_ml_kg_hr    0.053810
     bun_cr_ratio    0.053663
age_x_comorbidity    0.039529

💡 5/10 of top features are engineered.
   Caution: with AUROC near chance, these importances mostly reflect noise the trees fit.


## 6. Temporal Features from Vital Signs (Marcus's Sepsis Journey)

The book describes the features that made Marcus's sepsis model work: **6-hour rolling means and standard deviations**
for each vital sign, **trends (slope over the past 4 hours)**, and **delta features (current value minus value 6 hours ago)**,
plus the **shock index** (HR/SBP).

We simulate 24 hours of vital signs every 15 minutes (96 readings per patient). Every patient has their own baseline,
and patients who develop sepsis drift away from *their own* baseline in the final hours, as Marcus did
(HR 94 against a baseline of 72). This is synthetic data built to illustrate the idea.

In [ ]:
rng_ts = np.random.default_rng(7)
n_ts, n_steps = 600, 96                 # 600 patients x 24 h at 15-min intervals
per_hour = 4                            # readings per hour
vital_names = ['hr', 'sbp', 'temp', 'rr']

# Patient-specific baselines: the same raw value can be normal for one patient and abnormal for another
base = np.column_stack([rng_ts.normal(80, 12, n_ts), rng_ts.normal(125, 15, n_ts),
                        rng_ts.normal(36.9, 0.3, n_ts), rng_ts.normal(16, 2.5, n_ts)])
noise_sd = np.array([6, 8, 0.25, 2.0])
vitals = base[:, None, :] + rng_ts.normal(0, 1, (n_ts, n_steps, 4)) * noise_sd

sepsis_ts = rng_ts.random(n_ts) < 0.30
drift_per_hour = np.array([1.0, -1.0, 0.04, 0.25])   # HR up, SBP down, temp up, RR up
for i in np.where(sepsis_ts)[0]:
    onset_h = rng_ts.uniform(4, 10)                    # deterioration starts 4-10 h before prediction time
    hours_in = np.clip(np.arange(n_steps) / per_hour - (24 - onset_h), 0, None)
    vitals[i] += hours_in[:, None] * drift_per_hour * rng_ts.uniform(0.5, 1.5)

def temporal_features(v):
    """Features at prediction time (end of the 24 h window) from one patient's (n_steps, 4) array."""
    w6, w4 = 6 * per_hour, 4 * per_hour
    t4 = np.arange(w4) / per_hour
    feats = {}
    for j, name in enumerate(vital_names):
        x = v[:, j]
        feats[f'{name}_now'] = x[-1]
        feats[f'{name}_mean6h'] = x[-w6:].mean()
        feats[f'{name}_std6h'] = x[-w6:].std()
        feats[f'{name}_slope4h'] = np.polyfit(t4, x[-w4:], 1)[0]     # units per hour
        feats[f'{name}_delta6h'] = x[-1] - x[-w6 - 1]
    feats['shock_index_now'] = v[-1, 0] / v[-1, 1]
    return feats

df_ts = pd.DataFrame([temporal_features(vitals[i]) for i in range(n_ts)])
df_ts['sepsis'] = sepsis_ts.astype(int)
print(f"Synthetic vital-sign cohort: {n_ts} patients, {sepsis_ts.mean():.0%} develop sepsis")
print(f"Temporal features per patient: {df_ts.shape[1] - 1}")

# The same 6h rolling mean, computed along the whole series with pandas (24 readings = 6 h)
i_non = np.where(~sepsis_ts)[0][0]
i_sep = df_ts['hr_delta6h'].where(sepsis_ts).idxmax()     # the septic patient with the clearest HR rise
hours = np.arange(n_steps) / per_hour
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, i, label in [(axes[0], i_non, 'No sepsis'), (axes[1], i_sep, 'Develops sepsis (clearest HR rise)')]:
    hr = pd.Series(vitals[i, :, 0])
    ax.plot(hours, hr, alpha=0.4, label='HR (every 15 min)')
    ax.plot(hours, hr.rolling(24).mean(), linewidth=2, label='6 h rolling mean')
    ax.set_xlabel('Hours'); ax.set_ylabel('Heart rate (bpm)'); ax.set_title(label, fontweight='bold'); ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

raw_cols = [f'{v}_now' for v in vital_names]
temporal_cols = [c for c in df_ts.columns if c != 'sepsis']
y_ts = df_ts['sepsis']
cv = StratifiedKFold(5, shuffle=True, random_state=0)
lr_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

auc_raw = cross_val_score(lr_pipe, df_ts[raw_cols], y_ts, cv=cv, scoring='roc_auc')
auc_si = cross_val_score(lr_pipe, df_ts[raw_cols + ['shock_index_now']], y_ts, cv=cv, scoring='roc_auc')
auc_temp = cross_val_score(lr_pipe, df_ts[temporal_cols], y_ts, cv=cv, scoring='roc_auc')

print("="*70)
print("SEPSIS: CURRENT VALUES vs TEMPORAL FEATURES (5-fold CV AUROC, logistic regression)")
print("="*70)
print(f"Current vital signs only (4 features):        {auc_raw.mean():.3f} ± {auc_raw.std():.3f}")
print(f"+ shock index (5 features):                   {auc_si.mean():.3f} ± {auc_si.std():.3f}")
print(f"All temporal features ({len(temporal_cols)} features):         {auc_temp.mean():.3f} ± {auc_temp.std():.3f}")
print("="*70)
gain = auc_temp.mean() - auc_raw.mean()
n_better = int((auc_temp > auc_raw).sum())          # same 5 splits for both, so compare fold by fold
if n_better == len(auc_raw):
    print(f"✓ Temporal features add {gain:+.3f} AUROC and win in all {n_better} folds.")
    print("  Each patient's own trajectory (delta, slope) separates sepsis better than a single snapshot,")
    print("  because baselines differ between patients.")
else:
    print(f"➖ Temporal features changed AUROC by {gain:+.3f} and won in only {n_better}/{len(auc_raw)} folds.")
if auc_si.mean() - auc_raw.mean() < 0.01:
    print("  The shock index adds little here: a logistic model already sees HR and SBP separately.")
    print("  Ratios help most when they encode a relationship the model cannot easily build itself.")

## 7. Feature Selection: Filter, Wrapper, and Embedded Methods

More features are not always better. To see whether selection methods keep the useful ones, we add 10 pure-noise
features to the 21 temporal features and ask each method to choose. Selection runs **inside** cross-validation
(in the pipeline), otherwise the AUROC estimate would be optimistic.

- **Filter**: `SelectKBest(mutual_info_classif)` scores each feature alone (fast, ignores interactions)
- **Wrapper**: `RFE` repeatedly fits a model and drops the weakest feature (slower)
- **Embedded**: L1-penalised logistic regression (LASSO) shrinks unhelpful coefficients to exactly zero; tree importance (Section 5) is another embedded method

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFE, SelectFromModel

rng_sel = np.random.default_rng(1)
noise_cols = [f'noise_{k}' for k in range(10)]
X_sel = pd.concat([df_ts[temporal_cols],
                   pd.DataFrame(rng_sel.normal(size=(n_ts, 10)), columns=noise_cols)], axis=1)
k = 8

selectors = {
    'Filter: mutual information': SelectKBest(lambda X, y: mutual_info_classif(X, y, random_state=0), k=k),
    'Wrapper: RFE (logistic)': RFE(LogisticRegression(max_iter=1000), n_features_to_select=k),
    'Embedded: L1 logistic': SelectFromModel(LogisticRegression(penalty='l1', C=0.05, solver='liblinear')),
}

rows = []
for name, selector in selectors.items():
    pipe = make_pipeline(StandardScaler(), selector, LogisticRegression(max_iter=1000))
    auc = cross_val_score(pipe, X_sel, y_ts, cv=cv, scoring='roc_auc')
    pipe.fit(X_sel, y_ts)                                   # refit on all data only to list the chosen features
    chosen = X_sel.columns[pipe[1].get_support()]
    rows.append({'Method': name, 'n_selected': len(chosen),
                 'noise_kept': sum(c in noise_cols for c in chosen),
                 'CV AUROC': round(auc.mean(), 3), 'Selected': ', '.join(chosen)})
auc_all = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
                          X_sel, y_ts, cv=cv, scoring='roc_auc')
rows.append({'Method': 'No selection (all features)', 'n_selected': X_sel.shape[1], 'noise_kept': 10,
             'CV AUROC': round(auc_all.mean(), 3), 'Selected': '(all)'})
sel_results = pd.DataFrame(rows)

print(sel_results[['Method', 'n_selected', 'noise_kept', 'CV AUROC']].to_string(index=False))
for _, r in sel_results.iloc[:-1].iterrows():
    print(f"\n{r['Method']}: {r['Selected']}")

best = sel_results.loc[sel_results['CV AUROC'].idxmax()]
print(f"\n💡 Best CV AUROC: {best['Method']} ({best['CV AUROC']:.3f}); all features: {auc_all.mean():.3f}.")
small = sel_results.iloc[:-1]
if (small['CV AUROC'] >= auc_all.mean() - 0.01).all():
    print("   Every selected subset is within 0.01 of the full model: fewer features, no real loss.")
else:
    print(f"   Selection cost up to {auc_all.mean() - small['CV AUROC'].min():.3f} AUROC here: with {n_ts} patients and")
    print(f"   {X_sel.shape[1]} features the full logistic model copes with the noise; selection mainly buys a")
    print("   smaller, more interpretable model (and matters more when features outnumber patients).")
print(f"   Noise features kept: " + ", ".join(f"{m.split(':')[0]} {n}" for m, n in zip(small['Method'], small['noise_kept'])))
if (small['noise_kept'] == 0).all():
    print("   None of the three methods picked a pure-noise feature.")

## 8. Key Takeaways

### What We Learned

1. **Clinical Ratios**:
   - BUN/Cr ratio for pre-renal vs intrinsic AKI
   - eGFR for kidney function staging
   - Urine output per kg per hour
   - Ratios encode clinical knowledge

2. **Interaction Features**:
   - Diabetes × Creatinine (synergistic risk)
   - Elderly + Oliguria (high-risk combination)
   - Shock indicator flag (hypotension + tachycardia) vs the continuous shock index (HR/SBP)
   - Capture non-linear relationships

3. **Transformations**:
   - Log transforms for skewed distributions
   - Polynomial terms for U-shaped relationships
   - Reciprocals for inverse relationships

4. **Performance Impact**:
   - Always test incremental value on held-out data, with an uncertainty estimate (see the bootstrap CI above)
   - In this synthetic run the gain is small and both models are near chance; on real data, well-chosen
     clinical features often help most when data are limited
   - Feature importance is only meaningful when the model actually predicts well

5. **Temporal Features** (Section 6):
   - 6h rolling mean/std, 4h slope, 6h delta capture trajectories that a single value misses

6. **Feature Selection** (Section 7):
   - Filter (mutual information), wrapper (RFE), embedded (L1 / tree importance)
   - Run selection inside cross-validation, or the estimate is optimistic

### Clinical Considerations

- **Interpretability**: Clinically meaningful features are easier to validate and trust
- **Generalization**: Features based on physiology generalize across institutions
- **Missing data**: Derived features may have more missingness (requires careful handling)
- **Leakage risk**: Be careful not to use future information
- **Clinical validation**: Always review engineered features with domain experts

### Best Practices

1. **Start with clinical knowledge**: What do clinicians use for decision-making?
2. **Create interpretable features**: Avoid overly complex transformations
3. **Use established scores**: Charlson, MELD, APACHE, Framingham, etc.
4. **Test incremental value**: Compare base vs engineered features
5. **Feature importance**: Validate that engineered features are being used
6. **Domain validation**: Review with clinicians before deployment
7. **Document rationale**: Explain why each feature was created

### Connection to Journey Stories

**Marcus (Sepsis)**: Temporal features critical - rolling statistics, deltas, trends (Section 6 above; see also Notebook 7.3)

**David (CVD Risk)**: Framingham score incorporates age, sex, smoking, BP, cholesterol ratios (see Notebook 5.3)

**Priya (Melanoma)**: Image features - colour-histogram and contrast features (see Notebook 6.2)

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 4: Data Preparation)*

## Exercises

1. **MELD Score**: Implement the Model for End-Stage Liver Disease (MELD) score using bilirubin, INR, and creatinine. Compare its predictive power to raw features.

2. **Temporal Features**: In Section 6, change the windows (e.g. 2h vs 12h rolling statistics). Which window length helps most, and why might the best window differ between vital signs?

3. **Feature Selection**: Vary `C` in the L1 model of Section 7. How many noise features survive as the penalty weakens?

4. **Interaction Discovery**: Use a tree-based model to automatically discover interactions. Do they match clinical knowledge?

5. **Cross-Institution Validation**: Simulate two hospitals with different patient populations. Do engineered features generalize better than raw features?